In [2]:
from sqlalchemy import create_engine
import pandas as pd
import logging
from pathlib import Path

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def get_chat_data(db_path):
    """
    获取chat表的所有数据并转换为DataFrame
    
    Args:
        db_path (str): 数据库文件路径
    
    Returns:
        pd.DataFrame: 包含chat表数据的DataFrame
    """
    try:
        # 创建数据库连接
        engine = create_engine(f'sqlite:///{db_path}')
        
        # 读取chat表数据
        query = """
            select b.name,a.*
                from (
                SELECT * from chat  
                where meta!='{}') a 
                left join (
                select id,name from user 
                ) b
                on a.user_id=b.id;
        """
        df = pd.read_sql_query(query, engine)
        
        logger.info(f"成功读取chat表数据，共 {len(df)} 条记录")
        return df
        
    except Exception as e:
        logger.error(f"读取chat表数据时出错: {str(e)}")
        raise

if __name__ == "__main__":
    # 数据库文件路径
    db_path = "../backend/data/webui.db"
    
    # 获取数据
    chat_df = get_chat_data(db_path)
    
    # 显示数据基本信息
    print("\n数据基本信息:")
    print(chat_df.info())
    
    # 显示前几行数据
    print("\n数据预览:")
    print(chat_df.head())
    
    # 显示数据统计信息
    print("\n数据统计信息:")
    print(chat_df.describe()) 

2025-07-15 18:43:54,788 - INFO - 成功读取chat表数据，共 1122 条记录



数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1122 entries, 0 to 1121
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        1122 non-null   object
 1   id          1122 non-null   object
 2   user_id     1122 non-null   object
 3   title       1122 non-null   object
 4   share_id    0 non-null      object
 5   archived    1122 non-null   int64 
 6   created_at  1122 non-null   int64 
 7   updated_at  1122 non-null   int64 
 8   chat        1122 non-null   object
 9   pinned      1122 non-null   int64 
 10  meta        1122 non-null   object
 11  folder_id   0 non-null      object
dtypes: int64(4), object(8)
memory usage: 105.3+ KB
None

数据预览:
   name                                    id  \
0  dali  905a46d5-f70e-4f8c-9f7d-60e70a906d9c   
1  dali  aafca5ad-200c-4ef3-8df8-6d9cab6ef5ca   
2  dali  95987881-def0-4be4-b6d2-a311de87b786   
3    cz  8b375edf-e3ec-43b3-8245-689cd056fcd4   
4    cz  fde04d

In [3]:
import pprint   
import json 
with open("tmp.json","w") as f:
    f.write(json.dumps(json.loads(chat_df[0:1]['chat'].tolist()[0]),indent=4,ensure_ascii=False)) 

In [35]:
chat_data=[]
import json
import datetime 
## s时间戳转时间
def timestamp_to_datetime(timestamp):
    return datetime.datetime.fromtimestamp(timestamp)

for i,r in chat_df.iterrows():
    chat_=json.loads(r['chat'])
    user_id=r['user_id']
    user_name = r['name']
    chat_created_at=timestamp_to_datetime(r['created_at'])
    chat_updated_at=timestamp_to_datetime(r['updated_at'])
    
    if user_name in ['dali','cz',' cz','xuchengba']:
        continue

    chat_model=chat_.get("models",[])
    chat_title = chat_.get("title",None)
    chat_id= chat_.get("id",None)

    for history in chat_.get("history",{}).items():
        if history=={}:
            continue
        if history[0]=="messages":
            for id,hist in history[1].items():
                chat_data.append({
                    "chat_id":chat_id,
                    "chat_user_id":user_id,
                    "chat_title":chat_title,
                    "chat_model":chat_model,
                    "last_chat_model":chat_model[-1],
                    "chat_created_at":chat_created_at,
                    "chat_updated_at":chat_updated_at,
                    "user_name":user_name,
                    "role":hist.get("role"),
                    "model":hist.get("models"),
                    "message_id":id,
                    "parentId":hist.get("parentId"),
                    "last_child_id":hist.get("childrenIds",[])[-1] if hist.get("childrenIds",[]) else 0,
                    "childrenIds":hist.get("childrenIds"),
                    "created_at":hist.get("timestamp"),
                    "content":hist.get("content")
                })
            
    # chat_data.append(r['chat'])

In [36]:
import pandas as pd
chat_data_pd=pd.DataFrame(chat_data)
# chat_data_pd=chat_data_pd[chat_data_pd.role!="assistant"]
chat_data_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3985 entries, 0 to 3984
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   chat_id          3985 non-null   object        
 1   chat_user_id     3985 non-null   object        
 2   chat_title       3985 non-null   object        
 3   chat_model       3985 non-null   object        
 4   last_chat_model  3985 non-null   object        
 5   chat_created_at  3985 non-null   datetime64[ns]
 6   chat_updated_at  3985 non-null   datetime64[ns]
 7   user_name        3985 non-null   object        
 8   role             3983 non-null   object        
 9   model            1984 non-null   object        
 10  message_id       3985 non-null   object        
 11  parentId         2659 non-null   object        
 12  last_child_id    3985 non-null   object        
 13  childrenIds      3983 non-null   object        
 14  created_at       3983 non-null   float64

In [37]:
## 获取所有的回答
chat_data_pd_respond=chat_data_pd[chat_data_pd.role=="assistant"]
chat_data_pd_respond.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 1 to 3984
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   chat_id          1999 non-null   object        
 1   chat_user_id     1999 non-null   object        
 2   chat_title       1999 non-null   object        
 3   chat_model       1999 non-null   object        
 4   last_chat_model  1999 non-null   object        
 5   chat_created_at  1999 non-null   datetime64[ns]
 6   chat_updated_at  1999 non-null   datetime64[ns]
 7   user_name        1999 non-null   object        
 8   role             1999 non-null   object        
 9   model            0 non-null      object        
 10  message_id       1999 non-null   object        
 11  parentId         1999 non-null   object        
 12  last_child_id    1999 non-null   object        
 13  childrenIds      1999 non-null   object        
 14  created_at       1999 non-null   float64     

In [38]:
## 获取所有的问
chat_data_pd_query=chat_data_pd[chat_data_pd.role=="user"]
chat_data_pd_query.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1984 entries, 0 to 3983
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   chat_id          1984 non-null   object        
 1   chat_user_id     1984 non-null   object        
 2   chat_title       1984 non-null   object        
 3   chat_model       1984 non-null   object        
 4   last_chat_model  1984 non-null   object        
 5   chat_created_at  1984 non-null   datetime64[ns]
 6   chat_updated_at  1984 non-null   datetime64[ns]
 7   user_name        1984 non-null   object        
 8   role             1984 non-null   object        
 9   model            1984 non-null   object        
 10  message_id       1984 non-null   object        
 11  parentId         660 non-null    object        
 12  last_child_id    1984 non-null   object        
 13  childrenIds      1984 non-null   object        
 14  created_at       1984 non-null   float64     

In [39]:
show_respond_data = chat_data_pd_respond.drop(columns=['message_id'])
show_respond_data.rename(columns={"content":"respond_content","parentId":"message_id"},inplace=True)

chat_show_data=chat_data_pd_query.merge(show_respond_data[["message_id","respond_content"]],on = 'message_id')

In [49]:
chat_show_data.reset_index(drop=True,inplace=True)
chat_show_data.info(verbose=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2003 entries, 0 to 2002
Columns: 17 entries, chat_id to respond_content
dtypes: datetime64[ns](2), float64(1), object(14)
memory usage: 266.2+ KB


In [ ]:
## 统计全部的使用量
total_user_count = len(chat_show_data)

In [52]:
chat_show_data.groupby('user_name').count()

,chat_id,chat_user_id,chat_title,chat_model,last_chat_model,chat_created_at,chat_updated_at,role,model,message_id,parentId,last_child_id,childrenIds,created_at,content,respond_content
user_name,,,,,,,,,,,,,,,,
zongyuting-,109,109,109,109,109,109,109,109,109,109,98,109,109,109,109,109
义政妈,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1
刘桦映,2,2,2,2,2,2,2,2,2,2,0,2,2,2,2,2
华娟冰,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1
呼呼huhu～,98,98,98,98,98,98,98,98,98,98,3,98,98,98,98,98
地瓜妈,391,391,391,391,391,391,391,391,391,391,113,391,391,391,391,391
宋园梦,88,88,88,88,88,88,88,88,88,88,30,88,88,88,88,88
康复师X壹壹妈,545,545,545,545,545,545,545,545,545,545,76,545,545,545,545,545
康复师x殷老师,67,67,67,67,67,67,67,67,67,67,57,67,67,67,67,67


In [ ]:
chat_data_pd_query.groupby('last_chat_model').count()

In [ ]:
chat_data_pd.groupby('role').count()

In [ ]:
chat_data_pd.groupby("user_name").count()

In [48]:
chat_data_pd.groupby("user_name").count().sum()

chat_id            3985
chat_user_id       3985
chat_title         3985
chat_model         3985
last_chat_model    3985
chat_created_at    3985
chat_updated_at    3985
role               3983
model              1984
message_id         3985
parentId           2659
last_child_id      3985
childrenIds        3983
created_at         3983
content            3985
dtype: int64

In [50]:
chat_show_data.groupby("user_name").count().mean()

chat_id            111.277778
chat_user_id       111.277778
chat_title         111.277778
chat_model         111.277778
last_chat_model    111.277778
chat_created_at    111.277778
chat_updated_at    111.277778
role               111.277778
model              111.277778
message_id         111.277778
parentId            36.833333
last_child_id      111.277778
childrenIds        111.277778
created_at         111.277778
content            111.277778
respond_content    111.277778
dtype: float64